<h1><center>Laboratorio 5: La desperación de Mr. Lepin 🐼</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes


### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Leonardo Navarro
- Nombre de alumno 2: Matias Sweet


---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Antigravity, Cursor, etc.) restringido a consultas, documentación y corrección de errores. 
- **Importante**: **¡Recuerden fijar semillas!** Así podemos reproducir sus resultados.

## Descripción del laboratorio.

### Importamos librerias utiles 😸

In [ ]:
!uv add numpy pandas scikit-learn umap-learn plotly

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.base import BaseEstimator, TransformerMixin


def plot_dim_reductions(
    pca_proj: np.ndarray,
    tsne_proj: np.ndarray,
    umap_proj: np.ndarray,
    name: None | str = None,
    colors: None | np.ndarray = None,
) -> go.Figure:
    fig = make_subplots(rows=1, cols=3, subplot_titles=("PCA", "t-SNE", "UMAP"))

    for i, (proj, title) in enumerate(zip([pca_proj, tsne_proj, umap_proj], ["PCA", "t-SNE", "UMAP"], strict=True)):
        temp_fig = px.scatter(
            x=proj[:, 0],
            y=proj[:, 1],
            color=colors.astype(str) if colors is not None else None,
            title=title,
            # showlegend=(i == 0),
        )

        for trace in temp_fig.data:
            trace.showlegend = i == 0
            fig.add_trace(trace, row=1, col=i + 1)

    fig.update_layout(height=400, width=1200, title_text=name)
    return fig

# Segmentación de Clientes en Tienda de Retail 🛍️

<p align="center">
  <img width=300 src="https://s1.eestatic.com/2018/04/14/social/la_jungla_-_social_299733421_73842361_854x640.jpg">
</p>

## 1.1 Cargar Dataset

Mr. Lepin, en una nueva reunión, le cuenta a ud y su equipo que los resultados derivados del análisis exploratorio de datos presentaron una gran utilidad para la empresa y que tiene un gran entusiasmo por continuar trabajando con ustedes.
Es por esto, que Mr. Lepin les pide que cargue y visualicen algunas de las filas que componen el Dataset.
A continuación un extracto de lo parlamentado en la reunión:

    - Usted: Es un gran logro para nuestro equipo que usted haya encontrado excelente el EDA. ¿Qué tiene en mente ahora?
    - Mr. Lepin: Resulta que hace algún tiempo, mientras tomaba un mojito en una reunión de gerentes en Panamá, oí a un *chato* acerca de **LRMFP**, que es un modelo que permite personificar a los clientes a través de la fabricación de distintos atributos que describen a los clientes. Lo encontré es-tu-pendo ñatito. 
    - Usted: Ehh bueno. Investigaremos acerca de este modelo y veremos lo que podemos hacer.

Por ende, su siguiente tarea es calcular **LRMFP** sobre cada cliente y luego hacer un análisis de las características generadas. Para esto, el área de ventas les entrega un nuevo archivo llamado `retail_dataset.pickle`, quien posee los datos del DataFrame original limpios y listos para obtener las características solicitadas por Mr. Lepin.

In [2]:
df_retail = pd.read_pickle(
    "https://github.com/MDS7202/MDS7202/raw/refs/heads/main/recursos/2026-01/labs/lab6/retail_dataset.pickle"
)
df_retail = df_retail.astype(
    {
        "Invoice": str,
        "StockCode": str,
        "Description": str,
        "Customer ID": str,
        "Country": str,
    }
)
df_retail.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


El dataset contiene **541.909 transacciones** de una tienda de retail online del Reino Unido,
registradas entre diciembre de 2009 y diciembre de 2011. Cada fila corresponde a una línea
de ítem dentro de una factura (`Invoice`), con su descripción, cantidad, precio unitario,
fecha y cliente asociado.

**Columnas relevantes para LRMFP:**
- `InvoiceDate`: timestamp de la transacción (incluye hora) — se normaliza a fecha para Periodicity.
- `Invoice`: identificador de factura — agrupa ítems de una misma compra.
- `Quantity` y `Price`: permiten calcular el gasto total por ítem (`Quantity × Price`).
- `Customer ID`: clave de agrupación para construir el perfil por cliente.

> No se modifica `df_retail` en ningún paso posterior; todas las transformaciones se
> realizan sobre copias para preservar el estado original del dataset.

## 1.2 Creación de nuevas Caracteristicas [2 Puntos] 

Como ya se les comentó, Mr. Lepin está interesado en obtener las características **LRMFP**, para esto les señala que estas características se construyen en base a las siguientes definiciones:

- **Length (L)**: Intervalo de tiempo, en días, entre la primera y la última visita del cliente. Mientras más grande sea el valor, más fiel es el cliente.

- **Recency (R)**: Indica hace cuánto tiempo el cliente realizó su última compra. Notar que para este caso, mientras más grande es el valor, menos interés posee el usuario para repetir una compra en uno de los locales.

- **Monetary (M)**: El término “monetario” se refiere a la cantidad media de dinero gastada por cada visita del cliente durante el período de observación y refleja la contribución del cliente a los ingresos de la empresa.

- **Frequency (F)**: Se refiere al número total de visitas del cliente durante el periodo de observación. Cuanto mayor sea la frecuencia, mayor será la fidelidad del cliente. 

- **Periodicity (P)**: Representa si los clientes visitan las tiendas con regularidad.

$$Periodicity(n)=std(IVT_1, ..., IVT_n)$$

&nbsp;&nbsp; &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Donde $IVT$ denota el tiempo entre visitas y n representa el número de valores de tiempo entre visitas de un cliente.
 

$$IVT_i=date\_diff(t_{i+1},t_i)$$

En base a las definiciones señaladas, diseñe una función que permita obtener las características **LRMFP** recibiendo un DataFrame como entrada. Para esto, no estará permitido el uso de iteradores; utilicen todas las herramientas que les ofrece `pandas` para realizar esto.

Una referencia que les puede ser útil es el [documento original](https://www.researchgate.net/publication/315979555_LRFMP_model_for_customer_segmentation_in_the_grocery_retail_industry_a_case_study) en donde se propone este método.

**<u>Formato</u> del Resultado Esperado:**

| Customer ID | Length | Recency | Frequency | Monetary | Periodicity |
|------------:|-------:|--------:|----------:|---------:|------------:|
|   12346.0   |    294 |      67 |        46 |   -64.68 |        37.0 |
|   12347.0   |     37 |       3 |        71 |  1323.32 |         0.0 |
|   12349.0   |    327 |      43 |       107 |  2646.99 |        78.0 |
|   12352.0   |     16 |      11 |        18 |   343.80 |         0.0 |
|   12356.0   |     44 |      16 |        84 |  3562.25 |        12.0 |

**Respuesta:**

**Diseño de `custom_features`**:

Se implementa la función `custom_features` que transforma el DataFrame crudo en un perfil
**LRMFP** por cliente, siguiendo las definiciones del modelo original. La implementación
es completamente vectorizada con `pandas`.

**Decisiones de implementación:**

| Feature | Definición | Decisión clave |
|---|---|---|
| **Length** | Días entre primera y última visita | `max(date) - min(date)` por cliente, en días. |
| **Recency** | Días desde última compra hasta `max_date` del dataset | La referencia es el máximo global del dataset, no la fecha actual. |
| **Monetary** | Gasto **promedio por factura** | Se agrupa por `(Customer ID, Invoice)` antes de promediar. Error común: sumar todos los ítems directamente. |
| **Frequency** | Total de líneas de transacción | `count()` de filas por cliente. |
| **Periodicity** | Std de días entre visitas consecutivas | Se normaliza `InvoiceDate` a fecha sin hora antes de deduplicar, para que facturas del mismo día a distinta hora cuenten como una sola visita. Clientes con una sola visita → `std = NaN` → se rellena con 0. |

In [ ]:
def custom_features(dataframe_in: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula las características LRMFP por cliente.

    - Length     : días entre primera y última visita del cliente.
    - Recency    : días desde la última compra hasta la fecha máxima del dataset.
    - Monetary   : gasto PROMEDIO por factura (no por ítem).
    - Frequency  : número total de líneas de transacción del cliente.
    - Periodicity: desviación estándar de los días entre visitas consecutivas.
                   Clientes con una sola visita → 0.

    Parámetros
    ----------
    dataframe_in : pd.DataFrame
        DataFrame crudo con columnas Invoice, InvoiceDate, Quantity, Price, Customer ID.

    Retorna
    -------
    pd.DataFrame indexado por Customer ID.
    """
    df = dataframe_in.copy()
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

    # Fecha máxima del dataset (referencia para Recency)
    max_date = df["InvoiceDate"].max()

    # Agrupador base por cliente
    by_customer = df.groupby("Customer ID")["InvoiceDate"]

    # Length: días entre primera y última visita 
    length = (by_customer.max() - by_customer.min()).dt.days.rename("Length")

    # Recency: días desde la última compra hasta hoy (max del dataset) 
    recency = (max_date - by_customer.max()).dt.days.rename("Recency")

    # Frequency: total de líneas de transacción por cliente 
    frequency = df.groupby("Customer ID")["Invoice"].count().rename("Frequency")

    # Monetary: gasto promedio POR FACTURA 
    # Paso 1: total de cada factura (suma de Quantity * Price por Invoice)
    # Paso 2: promedio de esos totales por cliente
    df["TotalPrice"] = df["Quantity"] * df["Price"]
    invoice_total = df.groupby(["Customer ID", "Invoice"])["TotalPrice"].sum()
    monetary = invoice_total.groupby("Customer ID").mean().rename("Monetary")

    # Periodicity: std de días entre visitas consecutivas 
    # Normalizamos a fecha sin hora para que dos facturas en el mismo día
    # cuenten como una sola visita.
    df["VisitDate"] = df["InvoiceDate"].dt.normalize()
    visit_dates = (
        df[["Customer ID", "VisitDate"]]
        .drop_duplicates()
        .sort_values(["Customer ID", "VisitDate"])
    )
    # Diferencia en días entre visitas consecutivas del mismo cliente
    visit_dates["IVT"] = (
        visit_dates.groupby("Customer ID")["VisitDate"]
        .diff()
        .dt.days
    )
    # Std por cliente; una sola visita → NaN → rellenamos con 0
    periodicity = (
        visit_dates.groupby("Customer ID")["IVT"]
        .std()
        .rename("Periodicity")
        .fillna(0)
    )

    result = pd.concat([length, recency, monetary, frequency, periodicity], axis=1)
    result.index.name = "Customer ID"
    return result

In [4]:
features_df = custom_features(df_retail)
features_df.head()

,Length,Recency,Monetary,Frequency,Periodicity
Customer ID,,,,,
12346.0,196,164,33.896364,33,43.614982
12347.0,37,2,661.660000,71,0.000000
12348.0,0,73,222.160000,20,0.000000
12349.0,181,42,890.380000,102,101.823376
12351.0,0,10,300.930000,21,0.000000


**Análisis del resultado:**

El dataset contiene **4.314 clientes únicos**. Algunos patrones notables del `describe()`:

- **Length**: el 25% de los clientes tiene `Length = 0`, es decir, todas sus compras
  ocurrieron en un solo día. La mediana es de 105 días y el máximo es 373 días
  (equivalente a ~1 año de historia de compras).

- **Recency**: media de ~90 días. Hay clientes con `Recency = 0` (compraron en la fecha
  más reciente del dataset) y otros con `Recency = 373` (abandonados hace más de un año).

- **Monetary**: hay clientes con `Monetary = 0` (posiblemente devoluciones que cancelan
  el gasto), y el máximo de 11.880 sugiere clientes mayoristas. La distribución es
  sesgada a la derecha (media 376 > mediana ~285).

- **Frequency**: altamente sesgada — la mediana es 43 líneas pero el máximo es 5.568,
  indicando la presencia de clientes con comportamiento extremo.

- **Periodicity**: el 50% de los clientes tiene `Periodicity = 0` (1.499 con una sola
  visita + clientes con exactamente 2 visitas, cuyo std es NaN → 0). Esto refleja
  que una gran parte de la base de clientes es de baja frecuencia.

In [8]:
# Verificación de custom_features:
# Comprobamos propiedades esperadas sin conocer los valores exactos del dataset.

assert features_df.columns.tolist() == [
    "Length", "Recency", "Monetary", "Frequency", "Periodicity"
], "Las columnas no coinciden con el orden esperado."

assert features_df.index.name == "Customer ID", "El índice debe llamarse 'Customer ID'."

assert (features_df["Length"] >= 0).all(), "Length no puede ser negativo."
assert (features_df["Recency"] >= 0).all(), "Recency no puede ser negativo."
assert (features_df["Frequency"] > 0).all(), "Frequency debe ser al menos 1."
assert (features_df["Periodicity"] >= 0).all(), "Periodicity no puede ser negativo."

# Clientes con Length=0 deben tener Periodicity=0 (una sola fecha de visita)
single_visit = features_df[features_df["Length"] == 0]
assert (single_visit["Periodicity"] == 0).all(), (
    "Clientes con Length=0 deben tener Periodicity=0."
)

# Sin NaN en ninguna columna
assert features_df.isna().sum().sum() == 0, "No deben quedar NaN en el resultado."

print("Todas las verificaciones de custom_features pasaron.")
print(f"   Clientes únicos: {len(features_df)}")
print(f"   Clientes con una sola visita (Periodicity=0): {(single_visit['Periodicity']==0).sum()}")
features_df.describe()

Todas las verificaciones de custom_features pasaron.
   Clientes únicos: 4314
   Clientes con una sola visita (Periodicity=0): 1499


,Length,Recency,Monetary,Frequency,Periodicity
count,4314.000000,4314.000000,4314.000000,4314.000000,4314.000000
mean,133.936486,90.269124,376.198874,92.940890,18.805017
std,132.827714,96.943482,492.309973,198.883579,32.017882
min,0.000000,0.000000,0.000000,1.000000,0.000000
25%,0.000000,17.000000,179.400000,18.000000,0.000000
50%,105.000000,52.000000,284.855000,43.000000,0.000000
75%,253.750000,135.000000,421.450000,100.000000,26.596654
max,373.000000,373.000000,11880.840000,5568.000000,255.972655


## 1.3 Pipelines 👷

Finalmente *Mr. Lepin* le pregunta si sería posible realizar un pipeline para realizar una segmentación de los clientes con los nuevos datos generados, a lo que usted responde que **sí** y propone la utilización de k-means para la segmentación.

A continuación siga los pasos requeridos para obtener la segmentación de clientes.

### 1.3.1 Estandarizar Caracteristicas [0.5 puntos]

Construya una clase llamada ``MinMax()`` utilizando ``BaseEstimator`` y ``TransformerMixin`` para realizar una transformación de cada una de las columnas de un DataFrame utilizando ``ColumnTransformer()`` más tarde (tome como referencia el siguiente [enlace](https://sklearn-template.readthedocs.io/en/latest/user_guide.html#transformer)).


 Para esto considere que Min-Max escaler queda dada por la ecuación:

$$MinMax = \dfrac{x-min(x)}{max(x) - min(x)}$$

Con esto buscamos que los valores que componen a las columnas se muevan en el rango de valores $[0, 1]$.

**Respuesta:**

**Diseño de `MinMax`**:

Se implementa un transformador Min-Max **desde cero**, compatible con la API de scikit-learn
(`BaseEstimator` + `TransformerMixin`), lo que permite integrarlo en `Pipeline` y
`ColumnTransformer`.


**Principio de diseño clave:** el `fit()` aprende y guarda `min_` y `max_` **por columna**
sobre los datos de entrenamiento. El `transform()` aplica la fórmula usando esos valores
guardados, sin recalcular. Esto garantiza que al escalar nuevos datos (ej. test set),
se use la misma referencia que se usó para entrenar.

In [9]:
class MinMax(BaseEstimator, TransformerMixin):
    """
    Transformador Min-Max compatible con scikit-learn pipelines.

    Escala cada columna al rango [0, 1]:
        MinMax(x) = (x - min) / (max - min)

    El min y max se calculan en fit() sobre los datos de entrenamiento
    y se reutilizan en transform() sin recalcularse.
    Nota: sklearn.preprocessing.MinMaxScaler está prohibido en este lab.
    """

    def fit(self, X, y=None):
        """
        Aprende el mínimo y máximo de cada columna (eje 0).

        Parámetros
        ----------
        X : array-like (n_samples, n_features)
        y : ignorado (convención sklearn)
        """
        X = np.array(X, dtype=float)
        self.min_ = X.min(axis=0)   # mínimo por columna
        self.max_ = X.max(axis=0)   # máximo por columna
        return self

    def transform(self, X):
        """
        Aplica la normalización usando los valores guardados en fit().

        Parámetros
        ----------
        X : array-like (n_samples, n_features)

        Retorna
        -------
        np.ndarray con valores en [0, 1] para datos dentro del rango de entrenamiento.
        """
        X = np.array(X, dtype=float)
        return (X - self.min_) / (self.max_ - self.min_)

In [10]:
# Verificación de MinMax

# 1. Caso básico: escalar una columna [0, 5, 10] → [0.0, 0.5, 1.0]
X_simple = np.array([[0], [5], [10]], dtype=float)
scaler = MinMax().fit(X_simple)
result = scaler.transform(X_simple)
np.testing.assert_array_almost_equal(result.flatten(), [0.0, 0.5, 1.0])
print("Caso básico: [0, 5, 10] → [0.0, 0.5, 1.0]")

# 2. El mínimo se convierte en 0.0 y el máximo en 1.0
X_test = np.array([[1, 100], [3, 200], [5, 400]])
scaler2 = MinMax().fit(X_test)
out = scaler2.transform(X_test)
assert out.min(axis=0).tolist() == [0.0, 0.0], "El mínimo de cada columna debe ser 0."
assert out.max(axis=0).tolist() == [1.0, 1.0], "El máximo de cada columna debe ser 1."
print("Los extremos de cada columna resultan en 0.0 y 1.0.")

# 3. transform() usa los valores de fit(), NO los recalcula
#    Si transformamos datos fuera del rango, los valores pueden salir de [0, 1]
X_train = np.array([[0], [10]], dtype=float)
X_out_of_range = np.array([[20]], dtype=float)  # mayor que el máximo de entrenamiento
scaler3 = MinMax().fit(X_train)
resultado_oor = scaler3.transform(X_out_of_range)[0, 0]
assert resultado_oor == 2.0, (
    f"transform() debe usar min/max del fit: se esperaba 2.0, se obtuvo {resultado_oor}"
)
print("transform() reutiliza los valores de fit() (no recalcula).")

# 4. Compatibilidad con DataFrames (ColumnTransformer pasa arrays)
df_prueba = pd.DataFrame({"A": [2.0, 4.0, 6.0], "B": [10.0, 30.0, 50.0]})
scaler4 = MinMax().fit(df_prueba)
out4 = scaler4.transform(df_prueba)
assert out4.shape == (3, 2), "La forma del resultado debe ser (3, 2)."
print("Acepta DataFrames como entrada.")

# 5. Integración: el resultado sobre features_df está en [0, 1]
feature_cols = ["Length", "Recency", "Monetary", "Frequency", "Periodicity"]
scaler5 = MinMax().fit(features_df[feature_cols].values)
scaled = scaler5.transform(features_df[feature_cols].values)
assert scaled.min() >= 0.0, "Ningún valor escalado debe ser menor que 0."
assert scaled.max() <= 1.0, "Ningún valor escalado debe ser mayor que 1."
print("Integración con features_df: todos los valores en [0.0, 1.0].")

print("\n Todas las verificaciones de MinMax pasaron correctamente.")

Caso básico: [0, 5, 10] → [0.0, 0.5, 1.0]
Los extremos de cada columna resultan en 0.0 y 1.0.
transform() reutiliza los valores de fit() (no recalcula).
Acepta DataFrames como entrada.
Integración con features_df: todos los valores en [0.0, 1.0].

 Todas las verificaciones de MinMax pasaron correctamente.


**Resultado de las verificaciones:**

Los 5 tests confirman que la implementación es correcta:

1. **Correctitud numérica**: `[0, 5, 10]` escala a `[0.0, 0.5, 1.0]` exacto.
2. **Extremos**: el mínimo de cada columna resulta en `0.0` y el máximo en `1.0`.
3. **Uso de valores del fit**: al transformar un valor fuera del rango de entrenamiento
   (ej. 20 cuando el máximo de entrenamiento es 10), el resultado es `2.0`, no `1.0`.
   Esto confirma que `transform()` usa los parámetros guardados en `fit()`, no los
   recalcula sobre los datos nuevos.
4. **Compatibilidad**: acepta tanto `np.ndarray` como `pd.DataFrame` como entrada.
5. **Integración real**: al aplicar `MinMax` sobre las 5 columnas LRMFP de los 4.314
   clientes, todos los valores quedan en el rango `[0.0, 1.0]`.

### 1.3.2 Pipelines de Proyecciones [0.5 puntos]

Para comparar técnicas de reducción de dimensionalidad, realice **tres pipelines** distintos sobre los datos **LRMFP** usando los siguientes métodos:
- **PCA**
- **t-SNE**
- **UMAP**

Para cada pipeline, siga estos pasos:
1. Obtenga las características **LRMFP** desde el DataFrame `retail_dataset.pickle` utilizando la función ``custom_features`` creada anteriormente, junto a ``FunctionTransformer()``. Considere esto como el primer paso de su pipeline.
2. En segundo lugar, usando ``ColumnTransformer()``, aplique el MinMax scaler creado por usted sobre todas las columnas generadas en el paso anterior.
3. Finalmente, aplique el método de reducción de dimensionalidad correspondiente (PCA, t-SNE o UMAP) para obtener las 2 componentes más relevantes.

A continuación, grafique las proyecciones obtenidas de las tres técnicas en una sola figura comparativa.

**Respuesta:**

In [ ]:
pipeline_pca = ...
pipeline_tsne = ...
pipeline_umap = ...

In [ ]:
# Utilice este código para ejecutar las pipelines y graficar.

pca_proj = pipeline_pca.fit_transform(df_retail)
tsne_proj = pipeline_tsne.fit_transform(df_retail)
umap_proj = pipeline_umap.fit_transform(df_retail)

fig = plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="Reducción de Dimensionalidad", colors=None)
fig.show()

### 1.3.3 Análisis de los Loadings de PCA [0.5 puntos]
Antes de continuar con la etapa de clustering, analice los *loadings* (pesos o coeficientes) de las componentes principales obtenidas con PCA. 

Utilice el siguiente tutorial para visualizarlos: https://plotly.com/python/pca-visualization/

- Calcule y reporte los *loadings* de las dos primeras componentes principales.
- Interprete qué características (**LRMFP**) son más relevantes en cada componente.
- Visualice los *loadings* usando un gráfico de barras para cada componente.



In [ ]:
# Código para calcular loadings.

### Preguntas sobre loadings:

- ¿Qué son los loadings de PCA?

> Respuesta: ...

- ¿Qué información relevante obtiene sobre la estructura de los datos a partir de los *loadings* de PCA?

> Respuesta: ...

- ¿Existe alguna relación interesante entre las direcciones de las variables?

> Respuesta: ...

## 1.4 Clustering

### 1.4.1 Método del Codo [0.5 puntos]

Utilizando la clase creada para escalamiento, aplique el método del codo para visualizar cuál es el número de clusters que mejor se ajustan a los datos. Realice esto utilizando el algoritmo K-means dentro de un pipeline para un $k \in [1,20]$, donde k representa el número de clusters del k-means. Para la realización de esta sección y la próxima (1.4.2), considere los mismos pasos utilizados para el t-SNE, pero **permutando el algoritmo de reducción de dimensionalidad por k-means.**

**Respuesta:**

In [ ]:
from sklearn.cluster import KMeans  # noqa: F401

...

### Preguntas Método del Codo

- A través del gráfico obtenido, comente y justifique qué valor de k escogería para realizar el k-means.

> Respuesta: ...

- Le fue útil el método del codo para encontrar el número de clusters?

> Respuesta: ...

- Si no fue así, ¿qué otros métodos podría haber usado para encontrar un número óptimo de clusters?

> Respuesta: ...

### 1.4.2 Segmentación de Clientes con K-Means 🎁 [1 punto]

Por último, Mr. Lepin, impaciente de no entender lo que usted intenta explicarle, le solicita que por favor muestre algún resultado "visual y entendible" de los grupos encontrados.

En base a la elección de k realizada en la sección anterior, utilice este valor escogido y entrene un modelo de K-means utilizando el mismo pipeline de scikit-learn utilizado anteriormente.

Una vez ajustado los datos, genere una tabla con los promedios (o medianas) para cada uno de los atributos, agrupando estos por el clúster que pertenecen.

Finalmente, construya un heatmap de las características promedio de cada cluster para visualizar y comparar los perfiles de los grupos.

**Estadísticas de Referencia para K=6:**

Ud. debe calcularlas - Varían de ejecución en ejecución.


|         | Length  | Recency   | Frequency | Monetary | Periodicity |       |
|---------|---------|-----------|----------|-------------|-------|-------|
| Cluster |         |           |          |             |       |       |
|    0    |   258.8 |      45.2 |     76.1 |      1107.7 | 107.6 |   449 |
|    1    |    76.1 |     217.6 |     45.5 |       791.7 |  14.1 |   466 |
|    2    |   368.5 |       4.8 |   2715.0 |    226621.6 |   4.2 |     4 |
|    3    |    85.3 |      45.7 |     65.8 |      1047.0 |  10.5 |   987 |
|    4    |   347.2 |      15.9 |   1658.0 |     35829.3 |   8.0 |    25 |
|    5    |   298.0 |      29.8 |    183.8 |      3639.9 |  32.0 |  1188 |

In [ ]:
# Aquí calcule K-Means
...

In [ ]:
# Utilice la siguiente función para graficar k-means. kmeans_labels = clusters obtenidos por k-means.
plot_dim_reductions(pca_proj, tsne_proj, umap_proj, name="KMeans K=6", colors=kmeans_labels)

In [ ]:
# Aquí grafique el Heatmap
...

### Preguntas sobre K-Means: 

- ¿Se separaron bien los distintos clusters en cada visualización? 

> Respuesta: ...

- ¿Es posible observar agrupaciones coherentes?

> Respuesta: ...

- ¿Quedarían mejor más o menos clusters?

> Respuesta: ...

- ¿K-Means, dada la forma de las proyecciones, será el mejor método para clusterizar este dataset?¿Habrá algún otro mejor?

> Respuesta: ...

Y por último:

- Nombre a cada uno de los clusters según el comportamiento de sus miembros (ej. "C1: Compran poco pero con gran valor...") - Si es necesario, ajuste el número de clusters antes de responder.

> Respuestas: ...


Justifique su respuesta y no decepcione a Mr. Lepin.

## 1.5 Detección de Anomalías con DBSCAN [1 punto]
En esta sección, utilizará el algoritmo DBSCAN para identificar posibles anomalías (outliers) en los clientes del retail.

- Puede aplicar DBSCAN sobre las características originales escaladas (**LRMFP**) o sobre alguna de las proyecciones 2D (PCA, t-SNE o UMAP). Justifique su elección en las preguntas al final de la sección.
- Visualice los resultados usando `plot_dim_reductions`, mostrando los clusters y resaltando los outliers (label = -1) en las tres proyecciones (PCA, t-SNE, UMAP).

In [ ]:
from sklearn.cluster import DBSCAN  # noqa: F401

...

In [ ]:
# Utilice este código para graficar. dbscan_labels = clusters/outliers obtenidos por DBSCAN.
fig_dbscan = plot_dim_reductions(
    pca_proj,
    tsne_proj,
    umap_proj,
    name="DBSCAN - Detección de Anomalías",
    colors=dbscan_labels,
)
fig_dbscan.show()

### Preguntas sobre DBSCAN


1. ¿Por qué decidiste usar los datos originales completos o las proyecciones para aplicar DBSCAN? ¿Por qué no usaste la otra opción?

> Respuesta: ...

2. ¿Cómo elegiste los parámetros de DBSCAN (`eps`, `min_samples`)? ¿Probaste diferentes valores? ¿Cómo afectó esto los resultados?

> Respuesta: ...

3. ¿Tienen sentido los outliers encontrados según el contexto del negocio? ¿Qué interpretación le das a estos clientes? Analiza los datos con pandas si es necesario.

> Respuesta: ...

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por correo, Discord o U-cursos.

![Gracias Totales!](https://i.pinimg.com/originals/65/ae/27/65ae270df87c3c4adcea997e48f60852.gif "bruno")


<br>
<center>
<img src="https://i.kym-cdn.com/photos/images/original/001/194/195/b18.png" width=100 height=50 />
</center>
<br>